# 01 — Data Exploration (Week 2)

**Goal:** understand the CRMLS sold-listings data before building a model that predicts `ClosePrice`
for *any* California single-family address, on or off the market.

Scope (task doc): `PropertyType == "Residential"` and `PropertySubType == "SingleFamilyResidence"`.

Data: the `CRMLSSold*.csv` monthly files placed in `data/raw/` (≥ 6 months). Raw data is never committed.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from idx_avm import config

pd.set_option("display.max_columns", 80)
sns.set_theme(style="whitegrid")
np.random.seed(config.RANDOM_SEED)
print("Raw data dir:", config.RAW_DIR)

## 1. Load every monthly file

In [ ]:
from idx_avm.load import load_raw, filter_single_family

raw = load_raw()
print(f"{len(raw):,} rows x {raw.shape[1]} columns from {raw['SourceFile'].nunique()} files")
raw.groupby("SourceFile").size().rename("rows").to_frame()

## 2. Property type mix and the single-family filter

In [ ]:
display(raw["PropertyType"].value_counts(dropna=False).head(10))
display(raw.loc[raw["PropertyType"] == "Residential", "PropertySubType"].value_counts(dropna=False).head(15))

sfr = filter_single_family(raw)
print(f"Single-family residential: {len(sfr):,} rows ({len(sfr) / len(raw):.1%} of all rows)")

## 3. Schema and missingness

Columns with heavy missingness are candidates to drop; missingness in key features may itself be informative (best-practices §06).

In [ ]:
missing = sfr.isna().mean().sort_values(ascending=False).rename("pct_missing").to_frame()
missing["dtype"] = sfr.dtypes.astype(str)
print(f"Columns > 50% missing: {(missing['pct_missing'] > 0.5).sum()} of {len(missing)}")
missing.head(40)

In [ ]:
key_cols = [c for c in ["ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
                        "LotSizeSquareFeet", "YearBuilt", "Latitude", "Longitude", "GarageSpaces"] if c in sfr.columns]
sfr[key_cols].describe(percentiles=[0.005, 0.01, 0.25, 0.5, 0.75, 0.99, 0.995]).T

## 4. Target: `ClosePrice`

Prices are heavily right-skewed, so the model will be trained on `log(ClosePrice)`.

In [ ]:
price = sfr.loc[sfr["ClosePrice"] > 0, "ClosePrice"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(price.clip(upper=price.quantile(0.99)), bins=80)
axes[0].set(title="ClosePrice (clipped at p99)", xlabel="$")
axes[1].hist(np.log10(price), bins=80)
axes[1].set(title="log10(ClosePrice)", xlabel="log10 $")
plt.tight_layout(); plt.show()
print(f"Median ${price.median():,.0f} | mean ${price.mean():,.0f} | skew {price.skew():.2f}")
print(f"ClosePrice <= 0 or missing: {(sfr['ClosePrice'].isna() | (sfr['ClosePrice'] <= 0)).sum():,} rows")

## 5. Core features: LivingArea, Bedrooms, Bathrooms, LotSize

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
specs = [("LivingArea", "sq ft"), ("BedroomsTotal", "count"), ("BathroomsTotalInteger", "count"), ("LotSizeSquareFeet", "sq ft")]
for ax, (col, unit) in zip(axes.ravel(), specs):
    if col not in sfr.columns:
        ax.set_visible(False); continue
    s = sfr[col].dropna()
    if s.nunique() <= 15:
        s.value_counts().sort_index().plot.bar(ax=ax)
    else:
        ax.hist(s.clip(s.quantile(0.005), s.quantile(0.99)), bins=60)
    ax.set(title=f"{col}  (missing {sfr[col].isna().mean():.1%})", xlabel=unit)
plt.tight_layout(); plt.show()

In [ ]:
valid = sfr[(sfr["ClosePrice"] > 0) & (sfr["LivingArea"] > 0)]
sample = valid.sample(min(20_000, len(valid)), random_state=config.RANDOM_SEED)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(sample["LivingArea"], sample["ClosePrice"], s=3, alpha=0.3)
axes[0].set(xscale="log", yscale="log", xlabel="LivingArea (log)", ylabel="ClosePrice (log)", title="Price vs living area")
sns.boxplot(data=sample[sample["BedroomsTotal"].between(1, 7)], x="BedroomsTotal", y="ClosePrice", ax=axes[1], showfliers=False)
axes[1].set(title="Price by bedrooms (no fliers)")
plt.tight_layout(); plt.show()

## 6. Location

Location is the dominant driver of price in California; county and ZIP will be key categorical features.

In [ ]:
by_county = (sfr[sfr["ClosePrice"] > 0].groupby("CountyOrParish")["ClosePrice"]
             .agg(sales="size", median_price="median").sort_values("sales", ascending=False))
display(by_county.head(15))
if {"Latitude", "Longitude"} <= set(sfr.columns):
    geo = sample.dropna(subset=["Latitude", "Longitude"])
    geo = geo[geo["Latitude"].between(*config.CA_LAT) & geo["Longitude"].between(*config.CA_LON)]
    fig, ax = plt.subplots(figsize=(7, 7))
    sc = ax.scatter(geo["Longitude"], geo["Latitude"], c=np.log10(geo["ClosePrice"]), s=2, cmap="viridis")
    plt.colorbar(sc, label="log10 ClosePrice"); ax.set(title="Sales by location", xlabel="Longitude", ylabel="Latitude")
    plt.show()
print("Distinct cities:", sfr["City"].nunique(), "| distinct ZIPs:", sfr["PostalCode"].nunique())

## 7. Time: volume and median price by close month

This motivates the chronological split in Week 3 — market level drifts month to month.

In [ ]:
monthly = (sfr[sfr["ClosePrice"] > 0].assign(CloseMonth=lambda d: d["CloseDate"].dt.to_period("M"))
           .groupby("CloseMonth")["ClosePrice"].agg(sales="size", median_price="median"))
display(monthly)
fig, ax1 = plt.subplots(figsize=(11, 4))
monthly["sales"].plot.bar(ax=ax1, color="lightgray"); ax1.set_ylabel("sales")
ax2 = ax1.twinx(); ax2.plot(range(len(monthly)), monthly["median_price"], marker="o", color="C0"); ax2.set_ylabel("median $")
ax1.set_title("Monthly sales volume and median ClosePrice"); plt.tight_layout(); plt.show()

## 8. Correlations among candidate (non-leaky) numeric features

In [ ]:
num = [c for c in config.NUMERIC_FEATURES + ["YearBuilt", "ClosePrice"] if c in sfr.columns]
corr = sfr.loc[sfr["ClosePrice"] > 0, num].corr(method="spearman")
plt.figure(figsize=(9, 7)); sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Spearman correlation"); plt.show()

## 9. Leakage audit

Fields that only exist because a home was listed or sold (ListPrice, OriginalListPrice, DaysOnMarket,
PurchaseContractDate, ListingContractDate, …) are **excluded from model inputs**. They are unavailable for
off-market homes and sit so close to ClosePrice that they inflate accuracy. The high correlations below are
the reason they are dangerous, not a reason to use them.

In [ ]:
from idx_avm.features import leakage_report
leakage_report(sfr).sort_values("spearman_corr_with_ClosePrice", ascending=False, key=lambda s: s.abs())

## 10. Findings to carry into preprocessing

Fill in after running on the real data:

- **Rows:** total vs single-family; any months with unusually low volume (possible partial month).
- **Target:** strongly right-skewed → model `log(ClosePrice)`; count of `ClosePrice <= 0` / missing rows to drop.
- **Missingness:** which key features are missing and how much (e.g. `LotSizeSquareFeet`, `GarageSpaces`).
- **Implausible values:** zero living area, extreme bed/bath counts, YearBuilt out of range, coordinates outside CA.
- **Location:** county/ZIP medians differ by multiples → strongest signal; ZIP is high-cardinality → target encoding.
- **Leakage:** list-price family and DOM excluded by design.